# 05_sol: Concurrent Web Crawler

Contains:
- the same scenario as `05_mock`
- one complete reference implementation
- grading tests


In [ ]:
# Chunk overview: Prepare imports, fixtures, and helper scaffolding used by the solution.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Import required modules for this solution step.
from collections import deque
# Import required modules for this solution step.
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
# Import required modules for this solution step.
from copy import deepcopy
# Import required modules for this solution step.
from threading import Lock
# Import required modules for this solution step.
from typing import Any
# Import required modules for this solution step.
from urllib.parse import urldefrag, urlparse
# Import required modules for this solution step.
import time

# Assign computed data to a named variable for later use.
WEB_GRAPH = {
    # Execute this line as part of the solution flow.
    "https://docs.local/start": [
        # Execute this line as part of the solution flow.
        "https://docs.local/a#intro",
        # Execute this line as part of the solution flow.
        "https://docs.local/b",
        # Execute this line as part of the solution flow.
        "https://external.com/ignore",
    # Execute this line as part of the solution flow.
    ],
    # Execute this line as part of the solution flow.
    "https://docs.local/a": [
        # Execute this line as part of the solution flow.
        "https://docs.local/b",
        # Execute this line as part of the solution flow.
        "https://docs.local/c",
    # Execute this line as part of the solution flow.
    ],
    # Execute this line as part of the solution flow.
    "https://docs.local/b": [
        # Execute this line as part of the solution flow.
        "https://docs.local/c#part",
        # Execute this line as part of the solution flow.
        "https://docs.local/d",
    # Execute this line as part of the solution flow.
    ],
    # Execute this line as part of the solution flow.
    "https://docs.local/c": [
        # Execute this line as part of the solution flow.
        "https://docs.local/start",
    # Execute this line as part of the solution flow.
    ],
    # Execute this line as part of the solution flow.
    "https://docs.local/d": [],
# Execute this line as part of the solution flow.
}


# Define class `FakeHtmlParser` to organize related behavior.
class FakeHtmlParser:
    # Define `__init__` so this step is reusable and testable.
    def __init__(self, graph: dict[str, list[str]], delay_seconds: float = 0.0) -> None:
        # Assign computed data to a named variable for later use.
        self._graph = deepcopy(graph)
        # Assign computed data to a named variable for later use.
        self._delay_seconds = delay_seconds
        # Assign computed data to a named variable for later use.
        self._calls: list[str] = []
        # Assign computed data to a named variable for later use.
        self._lock = Lock()

    # Define `getUrls` so this step is reusable and testable.
    def getUrls(self, url: str) -> list[str]:
        # Check this condition to choose the correct branch.
        if self._delay_seconds:
            # Call this function to perform the next operation.
            time.sleep(self._delay_seconds)
        # Use context management for safe setup/cleanup.
        with self._lock:
            # Call this function to perform the next operation.
            self._calls.append(url)
        # Return the computed value for the caller.
        return deepcopy(self._graph.get(url, []))

    # Define `call_count` so this step is reusable and testable.
    def call_count(self) -> int:
        # Use context management for safe setup/cleanup.
        with self._lock:
            # Return the computed value for the caller.
            return len(self._calls)


In [ ]:
# Chunk overview: Implement the final reference solution in a clean, stepwise way.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Define `normalize_url` so this step is reusable and testable.
def normalize_url(url: str) -> str:
    # Assign computed data to a named variable for later use.
    clean, _ = urldefrag(url)
    # Assign computed data to a named variable for later use.
    parsed = urlparse(clean)
    # Check this condition to choose the correct branch.
    if clean.endswith("/") and parsed.path not in ("", "/"):
        # Return the computed value for the caller.
        return clean[:-1]
    # Return the computed value for the caller.
    return clean


# Define `crawl_single_thread` so this step is reusable and testable.
def crawl_single_thread(start_url: str, parser: Any) -> list[str]:
    # Assign computed data to a named variable for later use.
    start = normalize_url(start_url)
    # Assign computed data to a named variable for later use.
    host = urlparse(start).hostname
    # Assign computed data to a named variable for later use.
    visited: set[str] = {start}
    # Assign computed data to a named variable for later use.
    queue: deque[str] = deque([start])

    # Loop while this condition remains true.
    while queue:
        # Assign computed data to a named variable for later use.
        current = queue.popleft()
        # Iterate through items to process each element deterministically.
        for nxt in parser.getUrls(current):
            # Assign computed data to a named variable for later use.
            url = normalize_url(nxt)
            # Check this condition to choose the correct branch.
            if urlparse(url).hostname != host:
                # Execute this line as part of the solution flow.
                continue
            # Check this condition to choose the correct branch.
            if url in visited:
                # Execute this line as part of the solution flow.
                continue
            # Call this function to perform the next operation.
            visited.add(url)
            # Call this function to perform the next operation.
            queue.append(url)

    # Return the computed value for the caller.
    return sorted(visited)


# Define `crawl_multi_thread` so this step is reusable and testable.
def crawl_multi_thread(start_url: str, parser: Any, max_workers: int = 4) -> list[str]:
    # Assign computed data to a named variable for later use.
    start = normalize_url(start_url)
    # Assign computed data to a named variable for later use.
    host = urlparse(start).hostname
    # Assign computed data to a named variable for later use.
    visited: set[str] = {start}
    # Assign computed data to a named variable for later use.
    lock = Lock()

    # Use context management for safe setup/cleanup.
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Assign computed data to a named variable for later use.
        in_flight: dict[Any, str] = {executor.submit(parser.getUrls, start): start}

        # Loop while this condition remains true.
        while in_flight:
            # Assign computed data to a named variable for later use.
            done, _ = wait(set(in_flight), return_when=FIRST_COMPLETED)
            # Iterate through items to process each element deterministically.
            for future in done:
                # Call this function to perform the next operation.
                in_flight.pop(future)
                # Iterate through items to process each element deterministically.
                for nxt in future.result():
                    # Assign computed data to a named variable for later use.
                    url = normalize_url(nxt)
                    # Check this condition to choose the correct branch.
                    if urlparse(url).hostname != host:
                        # Execute this line as part of the solution flow.
                        continue
                    # Use context management for safe setup/cleanup.
                    with lock:
                        # Check this condition to choose the correct branch.
                        if url in visited:
                            # Execute this line as part of the solution flow.
                            continue
                        # Call this function to perform the next operation.
                        visited.add(url)
                    # Assign computed data to a named variable for later use.
                    in_flight[executor.submit(parser.getUrls, url)] = url

    # Return the computed value for the caller.
    return sorted(visited)


In [ ]:
# Chunk overview: Run checks that prove the implementation meets the problem contract.
# Why this chunk exists: it makes the solution easier to reason about under interview time pressure.
# Define `run_exam05_tests` so this step is reusable and testable.
def run_exam05_tests() -> None:
    # Assign computed data to a named variable for later use.
    expected = [
        # Execute this line as part of the solution flow.
        "https://docs.local/a",
        # Execute this line as part of the solution flow.
        "https://docs.local/b",
        # Execute this line as part of the solution flow.
        "https://docs.local/c",
        # Execute this line as part of the solution flow.
        "https://docs.local/d",
        # Execute this line as part of the solution flow.
        "https://docs.local/start",
    # Execute this line as part of the solution flow.
    ]

    # Assign computed data to a named variable for later use.
    parser_single = FakeHtmlParser(WEB_GRAPH, delay_seconds=0.0)
    # Assign computed data to a named variable for later use.
    single = crawl_single_thread("https://docs.local/start#home", parser_single)
    # Assert expected behavior to validate correctness.
    assert single == expected
    # Assert expected behavior to validate correctness.
    assert parser_single.call_count() == len(expected)

    # Assign computed data to a named variable for later use.
    parser_multi = FakeHtmlParser(WEB_GRAPH, delay_seconds=0.01)
    # Assign computed data to a named variable for later use.
    multi = crawl_multi_thread("https://docs.local/start#home", parser_multi, max_workers=4)
    # Assert expected behavior to validate correctness.
    assert multi == expected
    # Assert expected behavior to validate correctness.
    assert parser_multi.call_count() == len(expected)

    # Assert expected behavior to validate correctness.
    assert single == multi
    # Call this function to perform the next operation.
    print("05_mock tests passed")


# Call this function to perform the next operation.
run_exam05_tests()
